# 03 — Failure cases

Finds the false positives and false negatives automatically by comparing predictions against the
ground-truth labels on the validation split, and writes annotated images for
`docs/error_analysis.md` and the slide pack.

Colours: **blue** = detection the model made · **red** = false positive ·
**orange dashed** = false negative (missed).

**How to run:** Runtime → **Run all**. Dataset and weights both download from this repository's
release — no API key, no account, nothing from a local drive.

**Expected runtime:** ~3–4 min.

## 1. Setup

In [ ]:
!pip install -q ultralytics==8.4.155

import ultralytics, torch
print("ultralytics", ultralytics.__version__)
print("cuda", torch.cuda.is_available())

## 2. Dataset — keyless download

The dataset is a frozen export of Roboflow version 1, published as a release asset on this
repository. It downloads over plain HTTPS with **no API key and no account**, and the SHA256 check
proves it is byte-for-byte the file the published results were trained on.

Roboflow remains the annotation and versioning tool; section 2b keeps that path as a fallback, but
it does not run when the keyless dataset is already in place.

In [ ]:
import hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/caprijopi-alt/ppe-detection-yolov8/releases/download/v1.0/construction-safety-v1-yolov8.zip"
SHA256   = "9ae7044a16fc8d7364d52f23d023abde5e06da5d748ea126768358da636be7c4"
ROOT     = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]


already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    print("downloading 190 MB ...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    print("checksum ok")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path      = find_data_yaml(ROOT)
dataset_root  = cfg_path.parent
cfg           = yaml.safe_load(cfg_path.read_text())
cfg["path"]   = str(dataset_root)
cfg["train"]  = "train/images"
cfg["val"]    = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

DATA_YAML = str(cfg_path)
print(cfg)

### 2b. Roboflow path (secondary, optional)

Skipped automatically when the keyless dataset is present, so **Run all** never stops for a
credential. Kept only so the Roboflow version this export came from stays visible and re-fetchable.
The version number is pinned; `"latest"` is never used.

In [ ]:
from pathlib import Path

if not list(Path("/content/dataset").rglob("data.yaml")):
    %pip install -q roboflow==1.5.0
    try:
        from google.colab import userdata
        api_key = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        from getpass import getpass
        api_key = getpass("Roboflow API key (any account works for public Universe datasets): ")

    from roboflow import Roboflow
    rf = Roboflow(api_key=api_key)
    rf.workspace("caprijopi-hotmail-com").project("construction-safety-gsnvb-oz6um").version(1).download(
        "yolov8", location="/content/dataset_roboflow"
    )
else:
    print("Keyless dataset already present; skipping Roboflow download.")

In [ ]:
import os, urllib.request

WEIGHTS = "/content/best.pt"
WEIGHTS_URL = "https://github.com/caprijopi-alt/ppe-detection-yolov8/releases/download/v1.0/best.pt"

if not os.path.exists(WEIGHTS):
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS)
print(round(os.path.getsize(WEIGHTS)/1e6, 2), "MB")

from ultralytics import YOLO
model = YOLO(WEIGHTS)
NAMES = model.names
print(NAMES)

## 3. Match predictions to ground truth

An IoU of 0.5 is the same threshold the mAP@50 figure uses, so what this cell counts as a miss is
what the metric counts as a miss. Predictions are run at a low confidence so we can also see the
*near* misses — detections the model made but scored below the operating threshold.

In [ ]:
import numpy as np, yaml
from pathlib import Path

CONF_OPERATING = 0.35   # the threshold you actually deploy at
CONF_FLOOR     = 0.10   # look below it to spot near-misses
IOU_MATCH      = 0.50

root = dataset_root
val_images = sorted((root / "valid" / "images").glob("*"))
print(len(val_images), "validation images")

def load_gt(img_path, w, h):
    lbl = root / "valid" / "labels" / (img_path.stem + ".txt")
    out = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            if not line.strip():
                continue
            c, cx, cy, bw, bh = line.split()[:5]
            cx, cy, bw, bh = float(cx)*w, float(cy)*h, float(bw)*w, float(bh)*h
            out.append((int(c), cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2))
    return out

def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

false_pos, false_neg = [], []

for img_path in val_images:
    r = model.predict(str(img_path), conf=CONF_FLOOR, verbose=False)[0]
    h, w = r.orig_shape
    gts = load_gt(img_path, w, h)

    preds = []
    for box, cls, cf in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
        preds.append((int(cls), cf, box))
    preds.sort(key=lambda p: -p[1])

    used = set()
    matched_pred = set()
    for pi, (pc, cf, pbox) in enumerate(preds):
        best_i, best_iou = -1, 0.0
        for gi, (gc, *gbox) in enumerate(gts):
            if gi in used or gc != pc:
                continue
            v = iou(pbox, gbox)
            if v > best_iou:
                best_i, best_iou = gi, v
        if best_iou >= IOU_MATCH:
            used.add(best_i)
            matched_pred.add(pi)

    for pi, (pc, cf, pbox) in enumerate(preds):
        if pi not in matched_pred and cf >= CONF_OPERATING:
            false_pos.append({"img": img_path, "cls": NAMES[pc], "conf": cf, "box": pbox})

    for gi, (gc, *gbox) in enumerate(gts):
        if gi not in used:
            near = max([cf for pc, cf, pb in preds
                        if pc == gc and iou(pb, gbox) >= 0.3] + [0.0])
            false_neg.append({"img": img_path, "cls": NAMES[gc], "box": gbox, "near_conf": near})

print(f"{len(false_pos)} false positives at conf>={CONF_OPERATING}")
print(f"{len(false_neg)} false negatives")

## 4. Which classes are failing

Check this against the per-class recall table in the README — `no-helmet` should dominate the false negatives.

In [ ]:
from collections import Counter

print("FALSE POSITIVES by class")
for k, v in Counter(f["cls"] for f in false_pos).most_common():
    print(f"  {k:12s} {v}")

print("\nFALSE NEGATIVES by class")
for k, v in Counter(f["cls"] for f in false_neg).most_common():
    print(f"  {k:12s} {v}")

print("\nMissed entirely (model scored nothing at all there) vs near-misses (scored below threshold):")
cold = [f for f in false_neg if f["near_conf"] == 0]
warm = [f for f in false_neg if f["near_conf"] > 0]
print(f"  cold misses  {len(cold)}")
print(f"  near misses  {len(warm)}  <- these would be recovered by lowering the threshold")

## 5. Pick the six cases

Highest-confidence false positives are the most embarrassing and the most instructive. For false
negatives we prioritise `no-helmet`, since that is the safety-critical class and the one the README
flags as failing.

In [ ]:
PRIORITY_CLASS = "no-helmet"

fp_pick = sorted(false_pos, key=lambda f: -f["conf"])[:3]

fn_sorted = sorted(false_neg, key=lambda f: (f["cls"] != PRIORITY_CLASS, f["near_conf"]))
fn_pick = fn_sorted[:3]

print("FALSE POSITIVES to screenshot")
for i, f in enumerate(fp_pick, 1):
    print(f"  fp_{i:02d}  {f['cls']:10s} conf {f['conf']:.2f}  {f['img'].name}")

print("\nFALSE NEGATIVES to screenshot")
for i, f in enumerate(fn_pick, 1):
    tag = "cold miss" if f["near_conf"] == 0 else f"near miss (best {f['near_conf']:.2f})"
    print(f"  fn_{i:02d}  {f['cls']:10s} {tag:26s} {f['img'].name}")

## 6. Draw and save

In [ ]:
from PIL import Image, ImageDraw
import shutil

EV = Path("/content/evidence"); shutil.rmtree(EV, ignore_errors=True); EV.mkdir()

def annotate(case, kind, idx):
    im = Image.open(case["img"]).convert("RGB")
    d = ImageDraw.Draw(im)
    r = model.predict(str(case["img"]), conf=CONF_OPERATING, verbose=False)[0]

    # every prediction the model made, in blue
    for box, cls, cf in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
        d.rectangle(box, outline=(40, 120, 255), width=2)
        d.text((box[0]+3, box[1]+3), f"{NAMES[int(cls)]} {cf:.2f}", fill=(40, 120, 255))

    # the case itself, highlighted
    if kind == "fp":
        d.rectangle(case["box"], outline=(230, 30, 30), width=5)
        d.text((case["box"][0]+3, case["box"][1]-14),
               f"FALSE POSITIVE: {case['cls']} {case['conf']:.2f}", fill=(230, 30, 30))
    else:
        x1, y1, x2, y2 = case["box"]
        for off in range(0, int(x2-x1), 16):          # dashed top and bottom edge
            d.line([(x1+off, y1), (min(x1+off+8, x2), y1)], fill=(255, 140, 0), width=5)
            d.line([(x1+off, y2), (min(x1+off+8, x2), y2)], fill=(255, 140, 0), width=5)
        d.line([(x1, y1), (x1, y2)], fill=(255, 140, 0), width=5)
        d.line([(x2, y1), (x2, y2)], fill=(255, 140, 0), width=5)
        d.text((x1+3, y1-14), f"MISSED: {case['cls']}", fill=(255, 140, 0))

    out = EV / f"{kind}_{idx:02d}_{case['cls']}.png"
    im.save(out)
    return out

from IPython.display import Image as Show, display

saved = []
for i, f in enumerate(fp_pick, 1):
    saved.append(annotate(f, "fp", i))
for i, f in enumerate(fn_pick, 1):
    saved.append(annotate(f, "fn", i))

for p in saved:
    print(p.name)
    display(Show(str(p), width=640))

## 7. Download

Unzip into `results/evidence/` in the repo, then write the caption for each one in
`docs/error_analysis.md`. The caption is where the marks are — the image only shows *what* failed,
the caption has to say *why*.

In [ ]:
shutil.make_archive("/content/failure_cases", "zip", EV)
from google.colab import files
files.download("/content/failure_cases.zip")